# Summary Algorithm

In [ ]:
# The summary algorithm computes a lot of descriptive statistics. I suggest to have a
# brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-summary-py/docs/v6-summary-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a two (federated-)step algorithm:
#
# 1. Call `summary_per_data_station`
# 2. Call `variance_per_data_station`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the summary statistics for
# the entire federated dataset. In IDEA4RC, the summary statistics per data station are
# also required. So in this notebook we go through the following steps to obtain both
# the *global* (from the central part) and the *local* (from the
# `summary_per_data_station` call) summary statistics:
#
# 1. Create a new vantage6 task to execute the *summary* method (central part). This
#    central part will start the tasks `summary_per_data_station` and
#    `variance_per_data_station` (as you can see in the swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* summary statistics from the central part (the main call)
# 4. Retrieve the *local* summary statistics from the data stations (the
#    `summary_per_data_station` call that was made by the central part)
#


In [ ]:
import base64
import json
import requests

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 3

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATIONS = {org["id"]: org["name"] for org in response.json()["data"] if org["id"] in [1,5]}
ORGANIZATION_IDS = list(ORGANIZATIONS.keys())
ORGANIZATION_NAMES = list(ORGANIZATIONS.values())

ORGANIZATIONS

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 117
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 94

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "ghcr.io/iknl/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "summary"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [321]

In [ ]:
# Before we can start analysis the cohorts (dataframes) we need to check the variables
# that are available in the dataframes. You can use this endpoint whenever you want user
# to allow you to select variables.
# TODO the dtpyes might change in the future, so do not rely on them to heavily now. In
# the next version of the data extraction job we will likely provide you with either the
# `category` or `numeric` colum type (so that you can use them to select varables)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{DATAFRAME_IDS[0]}",
    headers=headers
)
VARIABLES = response.json()["columns"]
VARIABLES

In [ ]:
ORGANIZATION_IDS

In [ ]:
org_input = [
    {
        "id": ORGANIZATION_IDS[0], # Central task
        "arguments": base64.b64encode(
            json.dumps(
                {
                    # "columns": VARIABLES,
                    # "numeric_columns": NUMERIC_VARIABLES,
                    "organizations_to_include": ORGANIZATION_IDS # all participants
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

In [ ]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
print(response.json())
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

In [ ]:
# Get the results of the (central) task, thus the *global* summary statistics.
response_global = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)

# Again, since it is a central task we can obtain the [0]th element of the data list
result_global = [
    json.loads(base64.b64decode(result["result"]).decode("UTF-8")) 
    for result in response_global.json()["data"]
]
result_global

In [ ]:
result_local = []
for cohort in result_global[0].keys():
    result_local.append({cohort: result for result in result_global[0][cohort]['partials']})

# Visualization of results

The results of the Summary Algorithm will be used in the Data Preparation part of the RAVEN UI.<br>
For the visualization of the results of the Summary Algorithm a distinction should be made between numeric and categoric variables. Hence these will be discussed separately.

## Numeric variables
The visualizations for the numeric variables are:
1) a table that contains the statistics rows 'N', 'Mean', 'Min', 'Max', 'Missing' and a column per cohort.
2) PER cohort: bars for 'Q1 per center', bars for 'Median per center', bars for 'Q3 per center', bars for 'Missing per center'.

> Note: Choose a variable to visualize, as example we now visualize "age".

In [ ]:
cohort_names = sorted(list([key for key in result_global[0].keys() if not key.endswith("_partial_results")]), key=len)
cohort_names

In [ ]:
def plot_bars(cohort, variable, organizations, measure):
    medians = [round(result[cohort]['numeric'][variable][measure],2) for result in result_local]
    center_names = [ORGANIZATIONS[int(result[cohort]['organization_id'])] for result in result_local]

    labels = [f"{name} - {median}" for name, median in zip(center_names, medians)]
    plt.figure(figsize=(4, 0.5 * len(medians)))

    # Ensure background bars show up even if all medians are zero
    max_value = max(max(medians), 1)

    # Draw full-width light gray bars
    plt.barh(center_names, [max_value] * len(medians), color='#e4e9ec', height=0.5)

    # Overlay actual data bars
    bars = plt.barh(center_names, medians, color=['#1ab5e5','#275b83'], height=0.5)

    # Position labels consistently to the *left* of the bars, based on max_value
    label_offset = max_value * 0.05  # adjust spacing as needed
    label_x = -label_offset  # left of the bar start (which is at x=0)

    for bar, label in zip(bars, labels):
        plt.text(label_x, bar.get_y() + bar.get_height() / 2,
                label, va='center', ha='right', fontsize=9)

    plt.gca().set_axis_off()
    plt.show()

In [ ]:
variables = result_global[0][cohort_names[0]]['numeric'].keys() # this cell can be used to figure out which variables are present
variables

In [ ]:
# Choose the variable to visualize
VARIABLE = "year_of_birth"

In [ ]:
columns = sorted(cohort_names, key=len)
df = pd.DataFrame(columns=columns, index=['N', 'Mean', 'Min', 'Max', 'Missing'])
df.index.name = 'Statistics'

print("####################")
print(VARIABLE)
print("####################")

for cohort in cohort_names:
    df.loc['N', cohort] = int(result_global[0][cohort]['numeric'][VARIABLE]['count'])
    df.loc['Mean', cohort] = round(result_global[0][cohort]['numeric'][VARIABLE]['mean'], 1)
    df.loc['Min', cohort] = int(result_global[0][cohort]['numeric'][VARIABLE]['min'])
    df.loc['Max', cohort] = int(result_global[0][cohort]['numeric'][VARIABLE]['max'])
    df.loc['Missing', cohort] = f"{int(result_global[0][cohort]['numeric'][VARIABLE]['missing'])} ({round(result_global[0][cohort]['numeric'][VARIABLE]['missing']/result_global[0][cohort]['numeric'][VARIABLE]['count']*100, 1)})%"

display(df)

for i, cohort in enumerate(cohort_names):
    print(ORGANIZATIONS)
    print("")
    print(f"Cohort {i+1}: {cohort}")

    print("")
    print("Q1 per center")
    plot_bars(cohort, VARIABLE, ORGANIZATIONS, 'q_25')

    print("")
    print("Median per center")
    plot_bars(cohort, VARIABLE, ORGANIZATIONS, 'median')

    print("")
    print("Q3 per center")
    plot_bars(cohort, VARIABLE, ORGANIZATIONS, 'q_75')

    print("Missing per center")
    plot_bars(cohort, VARIABLE, ORGANIZATIONS, 'missing')

## Categoric variables

The visualizations for the categoric variables are:
1) A table that contains the categories (of a certain variable) vs the cohorts. This table has 3 visualization options: frequencies, percentages of total, row percentages.
2) PER cohort: a table that contains the categories vs the centers. This table has 4 visualization options: frequencies, percentages of total, row percentages, and column percentages.
<br><br>
> Note1: Choose a variable to visualize, as example we now visualize "fnclcc_grade".<br>

> Note2: The "Total" rows are a sum of the categories of that column, these do NOT include the missing.<br>

> Note3: The "Missing" rows show the number of missing values. The percentage is calculated by n_missing / (total+n_missing) * 100

In [ ]:
cohort_names = sorted(list(result_global[0].keys()), key=len)

### 1. Category vs Cohorts

In [ ]:
def add_full_row_border(styler, row_label, border="3px solid black"):
    """
    Add a horizontal border across the entire table (index + data columns)
    for the given row label in a pandas Styler.
    """
    df = styler.data
    row_pos = df.index.get_loc(row_label) + 1  # +1 because nth-child counts from 1

    # Apply styling to all cells in that HTML row
    return styler.set_table_styles(
        [
            {
                "selector": f"tbody tr:nth-child({row_pos})",
                "props": [( "border-top", border )]
            }
        ],
        overwrite=False
    )

def split_missing_row(F, missing_label='Missing'):
    """Return (F_wo_missing, miss_counts, miss_pct_as_float_per_col)."""
    F = F.copy()
    miss_str = F.loc[missing_label].astype(str)
    
    # Extract counts and % from "n (p%)"
    counts = miss_str.str.extract(r'^\s*(\d+)').astype(float)[0].fillna(0).astype(int)
    pcts   = miss_str.str.extract(r'\(([\d.]+)\s*%\)').astype(float)[0].fillna(0.0)
    F_wo = F.drop(index=missing_label)
    return F_wo, counts, pcts

def coerce_numeric_block(df_block):
    """
    Coerce a DataFrame block to numeric:
      - Converts None/'None'/'' to NaN
      - Uses pd.to_numeric(errors='coerce') column-wise
    """
    out = df_block.copy()
    # Replace common non-numeric placeholders with NaN
    out = out.where(~out.isin([None, 'None', '']), np.nan)
    # Coerce each column to numeric
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out

def to_col_percent(F, total_label='Total', missing_label='Missing', digits=1):
    """
    Column % view (excluding missing in denominators):
      - Category rows: each cell / column total (non-missing) * 100
      - Total row: 100%
      - Missing row: numeric %, p = Missing_col / (Total_col + Missing_col)
    Robust to None/'None'/NaN in numeric cells.
    """
    F_wo, miss_counts, _ = split_missing_row(F, missing_label)
    F_num = coerce_numeric_block(F_wo)

    if total_label not in F_num.index:
        raise ValueError(f"Expected a '{total_label}' row with column totals in the table.")

    col_totals = F_num.loc[total_label].astype(float)
    cat_rows = F_num.index.difference([total_label])

    denom = col_totals.replace(0, np.nan)  # avoid divide by zero for category rows
    pct = (F_num.loc[cat_rows].div(denom, axis=1) * 100).round(digits)

    # Total row = 100%
    pct.loc[total_label] = 100.0

    # Missing row: per-column denominator = Total_col + Missing_col (numeric, not string)
    miss_den = col_totals.fillna(0) + miss_counts.fillna(0)
    miss_pct = np.where(miss_den > 0, (miss_counts.fillna(0) / miss_den) * 100, 0.0)
    pct.loc[missing_label] = np.round(miss_pct, digits)

    return pct

def to_row_percent(F, total_label='Total', missing_label='Missing', digits=1, total_col_label='Total'):
    """
    Row % view with special handling when a column named `total_col_label` exists:
      - Category rows:
          * `Total` column = computed as (row sum across non-Total columns) / row sum across non-Total columns = 100 if denom>0 else 0
          * Other columns normalized to sum to 100% (denominator = sum across non-Total columns in that row)
      - Index `Total` row:
          * `Total` column computed from non-Total columns (not forced); other columns normalized to sum to 100% (denominator = sum across non-Total columns of the totals row)
      - Missing row:
          * numeric %, per column, p = Missing_col / (Total_col + Missing_col)

    Robust to None/'None'/NaN in numeric cells. Percentages rounded to `digits` decimals.
    """
    F_wo, miss_counts, _ = split_missing_row(F, missing_label)
    F_num = coerce_numeric_block(F_wo)

    if total_label not in F_num.index:
        raise ValueError(f"Expected a '{total_label}' row with column totals in the table.")

    has_total_col = total_col_label in F_num.columns
    col_totals = F_num.loc[total_label].astype(float)
    cat_rows = F_num.index.difference([total_label])

    if has_total_col:
        non_total_cols = [c for c in F_num.columns if c != total_col_label]

        # Category rows: normalize non-Total columns to 100 per row
        row_denoms_non_total = (F_num.loc[cat_rows, non_total_cols]
                                .sum(axis=1, skipna=True)
                                .replace(0, np.nan))
        pct_rows_non_total = (F_num.loc[cat_rows, non_total_cols]
                              .div(row_denoms_non_total, axis=0) * 100).round(digits)

        # Total column: computed as row_sum_non_total / row_sum_non_total = 100 if denom>0 else 0
        total_col_vals = np.where(row_denoms_non_total.fillna(0) > 0, 100.0, 0.0)
        pct_rows = pct_rows_non_total.copy()
        pct_rows[total_col_label] = total_col_vals

        # Reorder columns to original order
        pct_rows = pct_rows.reindex(columns=F_num.columns)

        # Index Total row: non-Total columns normalized to 100
        sum_non_total = float(col_totals[non_total_cols].sum(skipna=True))
        if sum_non_total > 0:
            total_pct_non_total = (col_totals[non_total_cols] / sum_non_total * 100).round(digits)
        else:
            total_pct_non_total = col_totals[non_total_cols] * 0.0

        # Total column: computed from non-Total columns (sum_non_total / sum_non_total = 100 if denom>0 else 0)
        total_col_cell = 100.0 if sum_non_total > 0 else 0.0

        pct_rows.loc[total_label, non_total_cols] = total_pct_non_total
        pct_rows.loc[total_label, total_col_label] = total_col_cell

    else:
        # No Total column present: classic row% across all columns
        row_denoms = F_num.loc[cat_rows].sum(axis=1, skipna=True).replace(0, np.nan)
        pct_rows = (F_num.loc[cat_rows].div(row_denoms, axis=0) * 100).round(digits)

        # Index Total row becomes column shares of grand non-missing total
        grand_total = float(F_num.loc[total_label].sum(skipna=True))
        if grand_total > 0:
            total_pct = (F_num.loc[total_label] / grand_total * 100).round(digits)
        else:
            total_pct = F_num.loc[total_label] * 0.0
        pct_rows.loc[total_label] = total_pct

    # Missing row: numeric %, per-column denominator = Total_col + Missing_col
    miss_den_col = col_totals.fillna(0) + miss_counts.fillna(0)
    miss_pct_col = np.where(miss_den_col > 0, (miss_counts.fillna(0) / miss_den_col) * 100, 0.0)
    pct_rows.loc[missing_label] = np.round(miss_pct_col, digits)

    return pct_rows

def to_total_percent(
    F,
    total_label='Total',
    missing_label='Missing',
    digits=1,
    total_col_label='Total'  # if a "Total" column exists, treat it specially
):
    """
    Total % view with special handling for a 'Total' column:

      Grand denominator:
        - sum over [category rows × non-Total columns] only.

      Category rows (all rows except the index `Total`):
        - Non-Total columns:      cell / grand_total * 100
        - Total column (if any):  (row sum across non-Total columns) / grand_total * 100

      Index `Total` row:
        - Non-Total columns:      (column total for that column) / grand_total * 100
        - Total column:           (sum of Total row across non-Total columns) / grand_total * 100
                                  (computed, not forced)

      Missing row:
        - numeric %, where p uses a GLOBAL denominator that excludes the 'Total' column:
            grand_den = grand_total + sum(missing over non-Total columns)

    Robust to None/'None'/''/NaN in numeric cells. Percentages rounded to `digits` decimals.
    """
    F_wo, miss_counts, _ = split_missing_row(F, missing_label)
    F_num = coerce_numeric_block(F_wo)

    if total_label not in F_num.index:
        raise ValueError(f"Expected a '{total_label}' row with column totals in the table.")

    has_total_col = (total_col_label in F_num.columns)
    base_cols = [c for c in F_num.columns if (not has_total_col) or (c != total_col_label)]
    cat_rows = F_num.index.difference([total_label])

    # Grand denominator over category rows × base columns
    grand_total = float(F_num.loc[cat_rows, base_cols].sum(skipna=True).sum()) if base_cols else 0.0

    # Prepare output
    pct = pd.DataFrame(index=list(cat_rows) + [total_label], columns=F_num.columns, dtype=float)

    # Category rows
    if grand_total > 0:
        if base_cols:
            pct.loc[cat_rows, base_cols] = (
                F_num.loc[cat_rows, base_cols] / grand_total * 100
            ).round(digits)
        if has_total_col:
            row_sum_non_total = F_num.loc[cat_rows, base_cols].sum(axis=1, skipna=True)
            pct.loc[cat_rows, total_col_label] = (row_sum_non_total / grand_total * 100).round(digits)
    else:
        if base_cols:
            pct.loc[cat_rows, base_cols] = 0.0
        if has_total_col:
            pct.loc[cat_rows, total_col_label] = 0.0

    # Index Total row
    if base_cols:
        if grand_total > 0:
            pct.loc[total_label, base_cols] = (
                F_num.loc[total_label, base_cols] / grand_total * 100
            ).round(digits)
        else:
            pct.loc[total_label, base_cols] = 0.0

    if has_total_col:
        total_row_sum_non_total = float(F_num.loc[total_label, base_cols].sum(skipna=True)) if base_cols else 0.0
        pct.loc[total_label, total_col_label] = round((total_row_sum_non_total / grand_total * 100) if grand_total > 0 else 0.0, digits)

    # Missing row: numeric %, global denominator excludes Total column
    total_missing_non_total = float(miss_counts.reindex(base_cols).fillna(0).sum()) if base_cols else 0.0
    grand_den = grand_total + total_missing_non_total
    if grand_den > 0:
        miss_pct_global = (miss_counts.fillna(0) / grand_den) * 100
    else:
        miss_pct_global = miss_counts.fillna(0) * 0.0

    pct.loc[missing_label] = np.round(miss_pct_global, digits)

    return pct



In [ ]:
variables = result_global[0][cohort_names[0]]['counts_unique_values'].keys() # this cell can be used to figure out which variables are present
variables

In [ ]:
# Choose the variable to visualize
VARIABLE = "diagnosis_code"

In [ ]:
print("####################")
print(VARIABLE, "- Frequencies")
print("####################")

columns = sorted(cohort_names, key=len)

l_categories = []
for cohort, result in result_global[0].items():
    l_categories = l_categories + list(result['counts_unique_values'][VARIABLE].keys())
categories_all = sorted(set(l_categories))
categories = [category for category in categories_all if category != 'N/A']

indices = sorted(categories) + ['Total', 'Missing']
df = pd.DataFrame(columns=columns, index=indices)
df.index.name = VARIABLE

for cohort in cohort_names:
    for category in categories_all:

        if category=='N/A' and category in result_global[0][cohort]['counts_unique_values'][VARIABLE]:
            df.loc['Missing', cohort] = int(result_global[0][cohort]['counts_unique_values'][VARIABLE][category])
        elif category=='N/A' and category not in result_global[0][cohort]['counts_unique_values'][VARIABLE]:
            pass
        elif category in result_global[0][cohort]['counts_unique_values'][VARIABLE]:
            df.loc[category, cohort] = int(result_global[0][cohort]['counts_unique_values'][VARIABLE][category])
            df.loc['Missing', cohort] = int(0)
        else:
            df.loc[category, cohort] = None
            df.loc['Missing', cohort] = int(0)

    df.loc['Total', cohort] = int(df.loc[categories, cohort].sum())
    df.loc['Missing', cohort] = f"{df.loc['Missing', cohort]} ({round(df.loc['Missing', cohort]/(df.loc['Missing', cohort]+df.loc['Total', cohort])*100, 1)})%"

df_freq = add_full_row_border(df.style, "Total", "1px solid black")
display(df_freq)

In [ ]:
print("####################")
print(VARIABLE, "- Column percentages")
print("####################")
F_col = to_col_percent(df, total_label='Total', missing_label='Missing', digits=1)
F_col = add_full_row_border(F_col.style.format(lambda v: f"{v:.1f}%" if isinstance(v, (int, float, np.floating)) else v), "Total", "1px solid black")
display(F_col)

In [ ]:
print("####################")
print(VARIABLE, "- Total percentages")
print("####################")
F_tot = to_total_percent(df, total_label='Total', missing_label='Missing', digits=1)
F_tot = add_full_row_border(F_tot.style.format(lambda v: f"{v:.1f}%" if isinstance(v, (int, float, np.floating)) else v), "Total", "1px solid black")
display(F_tot)

### 2. Category vs Centers

In [ ]:
variables = result_global[0][cohort_names[0]]['counts_unique_values'].keys() # this cell can be used to figure out which variables are present
variables

In [ ]:
# Choose the variable to visualize
VARIABLE = "life_status"

In [ ]:
cohort_names # this cell can be used to figure out which cohorts are present

In [ ]:
# Choose the cohort to visualize
COHORT = "mystifying_grothendieck"

In [ ]:
print("####################")
print(VARIABLE, "- Frequencies")
print("####################")

l_categories = []
for cohort, result in result_global[0].items():
    l_categories = l_categories + list(result['counts_unique_values'][VARIABLE].keys())
categories_all = sorted(set(l_categories))
categories = [category for category in categories_all if category != 'N/A']

cohort = COHORT
# for i, cohort in enumerate(cohort_names):
print(f"Cohort {cohort}")

columns = list(ORGANIZATIONS.values())+['Total']
indices = sorted(categories) + ['Total', 'Missing']
df_centers = pd.DataFrame(columns=columns, index=indices)
df_centers.index.name = VARIABLE

for result_center in result_local:
    org_id = int(result_center[cohort]['organization_id'])

    for category in categories_all:

        if category=='N/A' and category in result_center[cohort]['counts_unique_values'][VARIABLE]:
            df_centers.loc['Missing', ORGANIZATIONS.get(org_id)] = int(result_center[cohort]['counts_unique_values'][VARIABLE][category])
        elif category=='N/A' and category not in result_center[cohort]['counts_unique_values'][VARIABLE]:
            pass
        elif category in result_center[cohort]['counts_unique_values'][VARIABLE]:
            df_centers.loc[category, ORGANIZATIONS.get(org_id)] = int(result_center[cohort]['counts_unique_values'][VARIABLE][category])
            df_centers.loc['Missing', ORGANIZATIONS.get(org_id)] = int(0)
        else:
            df_centers.loc[category, ORGANIZATIONS.get(org_id)] = None
            df_centers.loc['Missing', ORGANIZATIONS.get(org_id)] = int(0)

    df_centers.loc['Total', ORGANIZATIONS.get(org_id)] = int(df_centers.loc[categories, ORGANIZATIONS.get(org_id)].sum())

indices_no_total_missing = [idx for idx in df_centers.index if idx not in ['Total', 'Missing']]
for category in indices_no_total_missing:
    if category!='N/A':
        df_centers.loc[category, 'Total'] = int(df_centers.loc[category, list(ORGANIZATIONS.values())].sum())
        df_centers.loc['Total', 'Total'] = int(df_centers.loc['Total', list(ORGANIZATIONS.values())].sum())
        df_centers.loc['Missing', 'Total'] = int(df_centers.loc['Missing'][list(ORGANIZATIONS.values())].astype(int).sum())

display(add_full_row_border(df_centers.style, "Total", "1px solid black"))
    

In [ ]:
print("####################")
print(VARIABLE, "- Column percentages")
print("####################")
print(f"Cohort {COHORT}")
F_col = to_col_percent(df_centers, total_label='Total', missing_label='Missing', digits=1)
F_col = add_full_row_border(F_col.style.format(lambda v: f"{v:.1f}%" if isinstance(v, (int, float, np.floating)) else v), "Total", "1px solid black")
display(F_col)

In [ ]:
print("####################")
print(VARIABLE, "- Row percentages")
print("####################")
print(f"Cohort {COHORT}")
F_row = to_row_percent(df_centers, total_label='Total', missing_label='Missing', digits=1)
F_row = add_full_row_border(F_row.style.format(lambda v: f"{v:.1f}%" if isinstance(v, (int, float, np.floating)) else v), "Total", "1px solid black")
display(F_row)

In [ ]:
print("####################")
print(VARIABLE, "- Total percentages")
print("####################")
print(f"Cohort {COHORT}")
F_tot = to_total_percent(df_centers, total_label='Total', missing_label='Missing', digits=1)
F_tot = add_full_row_border(F_tot.style.format(lambda v: f"{v:.1f}%" if isinstance(v, (int, float, np.floating)) else v), "Total", "1px solid black")
display(F_tot)